In [1]:
import os

In [2]:
print("ok")


ok


In [3]:
%pwd

'c:\\Users\\karti\\OneDrive\\Desktop\\CHATBOT\\End-to-end-Medical-Chatbot-Generative-AI\\research'

In [4]:
import os
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\karti\\OneDrive\\Desktop\\CHATBOT\\End-to-end-Medical-Chatbot-Generative-AI'

In [6]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


c:\ProgramData\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
#extract the data from the pdf
def load_pdf_file(data):
    loader= DirectoryLoader(data,glob="*.pdf",loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents


In [8]:
extracted_data = load_pdf_file(data='Data/')

In [10]:
#split the data into chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap  = 20,
        length_function = len,
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


In [35]:
text_chunks=text_split(extracted_data)
print(len(text_chunks))

5859


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings


In [13]:
def download_huggingface_embeddings():
    embeddings =HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [14]:
embeddings = download_huggingface_embeddings()

In [15]:
query_result = embeddings.embed_query("hello world")
print(len(query_result))

384


In [16]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
import os 
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name="medicalbot"

try:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        )
    )
    print(f"Index '{index_name}' created successfully")
except Exception as e:
    if "already exists" in str(e) or "ALREADY_EXISTS" in str(e):
        print(f"Index '{index_name}' already exists. Using existing index.")
    else:
        raise

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents= text_chunks,          
    embedding=embeddings,
    index_name=index_name
)
   
print("Documents successfully stored in Pinecone http://172.16.3.88:8081")


Documents successfully stored in Pinecone ✅


In [60]:
docsearch
retreiver = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})



In [61]:
retreived_docs = retreiver.invoke("What is Acne?")
retreived_docs


[Document(id='316a5547-aa11-42bc-915a-e52a6dd088bb', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='64f1bd47-9f74-433e-b575-3bcb3d5aeb75', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='7c72da47-906d-4805-815d-1f0ec71e29ce', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39, 'page_labe

In [11]:

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    
)


In [98]:
from langchain_classic.chains.retrieval import  create_retrieval_chain 
from langchain_classic.chains.combine_documents import create_stuff_documents_chain 

from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are a helpful medical assistant. "
    "Use the following context to answer the user's question. "
    "If you don't know the answer, just say that you don't know. "
    "Don't try to make up an answer. "
    "Use the following pieces of retrieved context to answer the question. "
    "Provide your answer in a concise manner.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)


In [99]:
question_answer_chain=create_stuff_documents_chain(llm,prompt)
rag_chain= create_retrieval_chain(retreiver,question_answer_chain)

In [105]:
response = rag_chain.invoke({"input": "What is acne?"})
print(response["answer"])


The provided context doesn't give a detailed explanation of what acne is. I don't know the answer based on the given information.


In [ ]:
response = rag_chain.invoke({"input": "ways to prevent acne?"})
print(response["answer"])


To prevent acne, consider the following:

1. Shampoo often and wear hair off face
2. Eat a well-balanced diet, avoiding foods that trigger flare-ups
3. Give dry pimples a limited amount of sun exposure (unless told otherwise)
4. Do not pick or squeeze blemishes
5. Reduce stress

These methods may help prevent or reduce acne.
